In [1]:
from importlib import reload

import os, sys
from pathlib import Path
import json
import time
from tqdm import tqdm

import pandas as pd 

# Can't use __file__ or __filename__ inside a Jupyter notebook
WORK_DIR = Path.cwd().parent

sys.path.append(str(WORK_DIR))

from src import (
    graph_gen,
    ollama_manager,
    video_tools,
    prompt_formatters,
    datasets,
)

from src.STAR_utils.visualization_tools import qa_visualization as qavis


from google import genai
from google.genai import types
from PIL import Image
from io import BytesIO


In [2]:
STAR_VSMALL = WORK_DIR / "data/datasets/STAR/STAR_annotations/STAR_val_small_200.json"

VIDEO_DIR = WORK_DIR / "data/datasets/action-genome/Charades_v1_480"
VIDEO_DUMP_DIR = WORK_DIR / "experiments/video_dump"


In [ ]:
star_vsmall_df = pd.read_json(STAR_VSMALL)
star_vsmall_df.iloc[94]


question_id                                          Sequence_T1_4719
question            Which object did the person put down after the...
video_id                                                        AS7SG
start                                                             0.0
end                                                              23.9
text                                                      The laptop.
question_program    [{'function': 'Situations', 'value_input': []}...
choices             [{'choice_id': 0, 'choice': 'The laptop.', 'ch...
situations          {'000174': {'rel_pairs': [['o000', 'o028'], ['...
Name: 94, dtype: object

In [ ]:
star_vsmall_df.iloc[94].to_dict()


{'question_id': 'Sequence_T1_4719',
 'question': 'Which object did the person put down after they held the sandwich?',
 'video_id': 'AS7SG',
 'start': 0.0,
 'end': 23.9,
 'text': 'The laptop.',
 'question_program': [{'function': 'Situations', 'value_input': []},
  {'function': 'Actions', 'value_input': []},
  {'function': 'Situations', 'value_input': []},
  {'function': 'Filter_Situations_with_Obj', 'value_input': ['sandwich']},
  {'function': 'Actions', 'value_input': []},
  {'function': 'Filter_Actions_with_Verb', 'value_input': ['hold']},
  {'function': 'Unique', 'value_input': []},
  {'function': 'Filter_After_Actions', 'value_input': []},
  {'function': 'Filter_Actions_with_Verb', 'value_input': ['put']},
  {'function': 'Query_Earliest_Action', 'value_input': []},
  {'function': 'Query_Objs', 'value_input': []}],
 'choices': [{'choice_id': 0,
   'choice': 'The laptop.',
   'choice_program': [{'function': 'Equal', 'value_input': ['laptop']}]},
  {'choice_id': 1,
   'choice': 'The f

In [ ]:
qavis.Vis_Video(star_vsmall_df.iloc[98].to_dict(), VIDEO_DIR, VIDEO_DUMP_DIR)


	Video Seg:  0.0s - 17.8s


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
gemini_sgg_df = pd.read_json(WORK_DIR / "notebooks/gemini_responses_small200_20250721_18:26:00.jsonl", lines=True)


In [ ]:
print(gemini_sgg_df.iloc[95]['chat_history'][3]['content'])


<stsg>
Frame 0:
<scene_graph>
door ---- on_the_right_of ---- wall
picture_frame ---- on_the_left_of ---- door
decorative_item ---- below ---- picture_frame
arm ---- on_the_left_of ---- door
</scene_graph>

Frame 1:
<scene_graph>
person ---- moving_towards ---- door
person ---- on_the_left_of ---- door
decorative_item ---- below ---- person
</scene_graph>

Frame 2:
<scene_graph>
person ---- close_to ---- door
person ---- interacting_with ---- door
person ---- on_the_left_of ---- door
right_hand ---- on ---- door
</scene_graph>

Frame 3:
<scene_graph>
person ---- opening ---- door
door ---- ajar ---- room
person ---- on_the_left_of ---- door
right_hand ---- on ---- door
</scene_graph>

Frame 4:
<scene_graph>
person ---- entering ---- room
door ---- open ---- room
person ---- on_the_left_of ---- room
</scene_graph>

Frame 5:
<scene_graph>
person ---- inside ---- room
door ---- open ---- room
</scene_graph>

Frame 6-9:
<scene_graph>
room ---- visible_through ---- doorway
room ---- dimly_li